In [1]:
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install faiss-cpu


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pickle
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\Abarna Studio\AppData\Local\Temp\ipykernel_4148\2341523403.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Chunked Documents

In [3]:
CHUNK_PATH = "../data/document_chunks.pkl"

with open(CHUNK_PATH, "rb") as file:
    chunks = pickle.load(file)

print(f"Loaded {len(chunks)} chunks.")

Loaded 14 chunks.


In [4]:
chunks[0]

Document(metadata={'source': 'FAQ', 'category': 'Policy'}, page_content='Question:\nOrder Confirmation and Large Item DeliveryAll purchases receive a 10-digit Order ID. For large appliances (refrigerators, washing machines), you will receive a call from our logistics partner within 48 hours to schedule a delivery window (4-hour slots). Delivery includes basic installation and removal of old appliances, unless explicitly declined at checkout. Check your spam folder if the Order Confirmation is missing after 30 minutes.')

In [5]:
print(chunks[0].page_content)

Question:
Order Confirmation and Large Item DeliveryAll purchases receive a 10-digit Order ID. For large appliances (refrigerators, washing machines), you will receive a call from our logistics partner within 48 hours to schedule a delivery window (4-hour slots). Delivery includes basic installation and removal of old appliances, unless explicitly declined at checkout. Check your spam folder if the Order Confirmation is missing after 30 minutes.


## Embedding Model

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6518.54it/s]


## Embeddings and Build FAISS

In [7]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


In [8]:
print("Number of Indexed Chunks:", vectorstore.index.ntotal)

Number of Indexed Chunks: 14


## Save Vector Database

In [9]:
VECTOR_PATH = "../vectorstore/faiss_index"
vectorstore.save_local(VECTOR_PATH)
print("FAISS Database Saved Successfully")

FAISS Database Saved Successfully


## Reload the Database

In [10]:
db = FAISS.load_local(
    VECTOR_PATH,
    embedding_model,
    allow_dangerous_deserialization=True
)
print("FAISS Database Loaded Successfully")

FAISS Database Loaded Successfully


## Test Semantic Retrieval

In [11]:
query = "How do I return a product?"
results = db.similarity_search(query, k=3)
print("Retrieved Documents:", len(results))

Retrieved Documents: 3


In [12]:
for i, doc in enumerate(results):
    print("="*80)
    print(f"Result {i+1}")
    print()
    print(doc.page_content)

Result 1

Company Knowledge Base Policies
Return Policy
Products may be returned within 30 days of delivery. Items should be in original condition and
include all accessories. Damaged, defective, or incorrect items are eligible for free return pickup.
Refunds are processed after inspection.
Refund Policy
Refunds are typically processed within 5 to 7 business days after return approval. Refunds are
issued to the original payment method. Cash-on-delivery refunds are transferred to a registered
bank account.
Result 2

Question:
60-Day Return and Repair PolicySmall electronics are eligible for a 60-day return in original condition. Large appliances are covered by a 30-day return window but are first subject to an on-site service repair attempt. If the repair fails, a full refund or exchange is issued. All returns require an RMA number generated via the Returns Portal.
Result 3

Answer:
60-Day Return and Repair PolicySmall electronics are eligible for a 60-day return in original condition. 

## Test Multiple Queries

In [13]:
queries = [
    "How can I reset my password?",
    "When will I receive my refund?",
    "How do I claim warranty?",
    "Can I pay using UPI?",
    "How do I contact technical support?"
]
for q in queries:
    print("="*80)
    print("Query:", q)
    docs = db.similarity_search(q, k=1)
    print(docs[0].page_content)
    print()

Query: How can I reset my password?


Question:
60-Day Return and Repair PolicySmall electronics are eligible for a 60-day return in original condition. Large appliances are covered by a 30-day return window but are first subject to an on-site service repair attempt. If the repair fails, a full refund or exchange is issued. All returns require an RMA number generated via the Returns Portal.

Query: When will I receive my refund?
Company Knowledge Base Policies
Return Policy
Products may be returned within 30 days of delivery. Items should be in original condition and
include all accessories. Damaged, defective, or incorrect items are eligible for free return pickup.
Refunds are processed after inspection.
Refund Policy
Refunds are typically processed within 5 to 7 business days after return approval. Refunds are
issued to the original payment method. Cash-on-delivery refunds are transferred to a registered
bank account.

Query: How do I claim warranty?
Answer:
Standard and Extended Warranty CoverageAll products carry a sta

## Using a Retriever

In [14]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)
print("Retriever Created Successfully")

Retriever Created Successfully


In [15]:
query = "What payment methods are accepted?"
documents = retriever.invoke(query)
print("Retrieved Documents:", len(documents))

Retrieved Documents: 3


In [16]:
for doc in documents:
    print("="*80)
    print(doc.page_content)

bank account.
Shipping Policy
Standard shipping takes 3 to 7 business days. Express shipping may be available in select
locations. Delivery timelines may vary due to weather, holidays, or courier disruptions.
Payment Policy
Customers may pay using UPI, debit cards, credit cards, net banking, and supported digital wallets.
All payment transactions are encrypted and processed through secure gateways.
Cancellation Policy
All payment transactions are encrypted and processed through secure gateways.
Cancellation Policy
Orders can be cancelled before shipment dispatch. Once an order is shipped, cancellation may not
be possible. Eligible cancellations receive a full refund according to the refund policy.
Membership & Rewards Policy
Members earn reward points on eligible purchases. Gold members receive additional benefits and
Members earn reward points on eligible purchases. Gold members receive additional benefits and
exclusive offers. Membership can be upgraded, renewed, or cancelled through

## Verify Metadata

In [17]:
documents[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-07-16T07:19:31+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-07-16T07:19:31+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': '../data/company_policies.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1'}